# MMA Fight Analyzer — Kaggle training runner

Same pipeline as `colab_train.ipynb`; all logic lives in the repo's `scripts/` and `src/mma/`.

**Before running — notebook settings (right side panel):**
- Accelerator: **GPU P100** (or T4 x2 — the code uses one GPU)
- Internet: **ON** (needs a phone-verified Kaggle account)

Only `/kaggle/working` survives a *Save Version* run — outputs land there automatically.
You can also download individual files from the Output panel at any time.

In [ ]:
!nvidia-smi -L

In [ ]:
%cd /kaggle/working
!git clone https://github.com/Maximilianb1/mma-fight-analyzer.git
%cd mma-fight-analyzer
# Kaggle preinstalls torch/torchvision/sklearn/etc. — only add what's missing,
# so pip doesn't touch the CUDA torch build
!pip install -q ultralytics gdown

In [ ]:
!python scripts/download_data.py       # Drive -> data/raw (~10-20 min)

In [ ]:
!python scripts/preprocess.py          # frames + identity masks (~15 min on GPU)

In [ ]:
!python scripts/train_gate.py          # skip if you restore a previous gate.pt into outputs/gate/

In [ ]:
# If the session dies mid-run, rerun with --folds i,j for the missing folds only.
!python scripts/train_phase.py --model r2plus1d

In [ ]:
!python scripts/train_phase.py --model lstm

In [ ]:
!python scripts/evaluate.py
!python scripts/diagnose_pressure.py   # straight-vs-swapped test should be clean now

In [ ]:
# Deployment model for the demo (run once the winner is chosen)
# !python scripts/train_phase.py --model r2plus1d --final

In [ ]:
# Bundle results for easy download from the Output panel
!zip -r -q /kaggle/working/outputs_round2.zip outputs
# Optional: persist the preprocessed cache so future sessions skip preprocess —
# after this run, create a private Kaggle Dataset from /kaggle/working/cache_round2.zip
# and attach it to the notebook (then: unzip it into data/cache instead of preprocessing).
!zip -r -q /kaggle/working/cache_round2.zip data/cache
!ls -lh /kaggle/working/*.zip